# T9b — TRL/PEFT vs torchtune: minh họa 4 lớp chặn + round-trip convert

Notebook minh họa vì sao T9b không thể "load PEFT adapter ngoài vào torchtune" bằng yaml thuần
(Decision #36 thay bằng #37). Chạy được trên CPU (dry-run), không cần GPU/token.

- Phần A: chứng minh 4 lớp chặn bằng đọc source torchtune 0.6.0.
- Phần B: demo convert ngược PEFT→torchtune + round-trip self-check (như code thật).
- Phần C (optional, cần GPU+HF token): load model 2 đường để xem khác biệt.

In [ ]:
# Setup — cài nhanh nếu chạy trên máy trống (Colab/pod)
import subprocess, sys
for pkg in ['torch', 'torchtune==0.6.0', 'safetensors', 'huggingface_hub']:
    try:
        __import__(pkg.split('==')[0].replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
import torch, torchtune, inspect
print('torch', torch.__version__, '| torchtune', torchtune.__version__, '| cuda:', torch.cuda.is_available())

## A. 4 lớp chặn — minh họa bằng source pinned

In [ ]:
from torchtune.orchestration import _utils
src = inspect.getsource(_utils.get_adapter_checkpoint_path)
print(src)
# Chặn #3: hàm này trả None ngay nếu should_load_recipe_state=False,
# bất kể yaml có khai báo adapter_checkpoint hay không.

In [ ]:
from torchtune.training.training import Trainer  # noqa
import torchtune.recipe_main as _  # noqa
import pathlib, inspect as ins
import torchtune
rp = pathlib.Path(ins.getfile(torchtune)).parent / 'recipes' / 'lora_dpo_single_device.py'
print(rp)
# In đúng dòng gate trong recipe setup():
for i, line in enumerate(rp.read_text().splitlines(), 1):
    if 'ADAPTER_KEY' in line or 'resume_from_checkpoint' in line:
        print(f'{i:4d}: {line}')

Chặn #1: recipe chỉ truyền `ADAPTER_KEY` vào `_setup_model` khi `resume_from_checkpoint=True`;
`False` → adapter PEFT ngoài bị **bỏ qua âm thầm**, LoRA random init.
Chặn #2: `resume_from_checkpoint=True` kéo theo `should_load_recipe_state=True` → đòi file
`recipe_checkpoint` (optimizer, epoch...) mà adapter PEFT của Meta không có.
Chặn #4: không hề có hàm convert ngược — xem cell dưới.

## B. Convert ngược PEFT → torchtune + round-trip check (demo mini, CPU)

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'tools' / 'pod_setup'))
from convert_peft_adapter_to_torchtune import peft_to_tune_adapter_weights, _round_trip_check

# State dict giả đúng shape Llama-3.1-8B (q_proj/v_proj rank 64), đã permute như PEFT format
fake = {k: torch.randn(4, 4) for k in [
    'base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight',
    'base_model.model.model.layers.0.self_attn.q_proj.lora_B.weight',
]}
converted = peft_to_tune_adapter_weights(fake)
print('PEFT keys     :', list(fake.keys()))
print('torchtune keys:', list(converted.keys()))
_round_trip_check(fake, converted)
print('[convert] Round-trip OK — khớp tuyệt đối')

Round-trip dùng **chính hàm xuôi thật** của torchtune (`tune_to_peft_adapter_weights`) làm
oracle: convert ngược kết quả phải tái sinh đúng state dict PEFT gốc, lệch thì từ chối ghi file.
Đây là cổng chặn duy nhất cho rủi ro silent-corrupt (permute sai chiều / nhầm key).

## C. (Optional, cần GPU + HF token) Load model 2 đường để xem khác biệt

In [ ]:
if not torch.cuda.is_available():
    print('Không có GPU — bỏ qua cell này. Chạy trên pod/Colab khi cần.')
else:
    # (1) TRL/PEFT path (Meta's original): base + adapter qua peft.PeftModel
    #     model = PeftModel.from_pretrained(base, 'facebook/Meta-SecAlign-8B')
    # (2) torchtune path (T9b): build lora_llama3_1_8b + nạp converted adapter qua
    #     lora_dpo_single_device_t9b.py recipe (manual_adapter_checkpoint)
    # Xem docs/reports-on-t9b.md §4 cho kết quả smoke thật trên Colab T4.
    print('Xem manual-t9b.md Phase 0-1 để chạy thật')

**Kết luận:** torchtune 0.6.0 không hỗ trợ load PEFT adapter ngoài bằng yaml → T9b dùng
convert ngược tự viết + gate bằng round-trip, và recipe `lora_dpo_single_device_t9b.py` patch 1
điểm để nạp adapter đã convert (Decision #37). File: `external/meta_secalign_artifacts/helpers/`.